# TalentIQ - Training Pipeline Guide

This notebook explains everything that happens inside `train.py`. It walks through the full ML pipeline from loading clean data to saving trained models and generating evaluation reports.

## Overview of the pipeline

The training pipeline runs through 6 steps in order. First the data is loaded and split into train and test sets. Then a preprocessor is built to encode and scale all features. After that the preprocessor is applied and SMOTE is used to balance the training data. Three models are then trained with hyperparameter search. All three are evaluated and compared. Finally the best model is saved along with plots and reports.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## Step 1 - Load and split the data

`load_and_prepare()` starts by loading the clean preprocessed data and calling `engineer_features()` to add the 7 new columns from the preprocessing step.

The data is then split into X (all input features) and y (the Attrition target column). Both are further split into train and test sets using an 80/20 ratio. We train on 80% and test on the remaining 20%, which is data the model has never seen. Testing on training data would make the model look great but fail in real use.

We use `stratify=y` to make sure both the train and test sets have the same ratio of 0s and 1s. Without this, random chance might put all the rare attrition cases into one set, which would make the evaluation unfair. The split files are saved to `data/splits/` for traceability.

In [ ]:
from sklearn.model_selection import train_test_split

# Load data
try:
    df = pd.read_csv('../data/processed/processed_full.csv')
except FileNotFoundError:
    df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
    df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Total rows: {len(df)}')
print(f'Train size: {len(X_train)} rows')
print(f'Test size: {len(X_test)} rows')
print(f'\nAttrition in train: {y_train.mean():.2%}')
print(f'Attrition in test: {y_test.mean():.2%}')
print('\nStratification check: both sets have approximately the same attrition rate.')

## Step 2 - Building the preprocessor

Machine learning models cannot handle raw text so all features must be converted to numbers. `build_preprocessor()` creates a `ColumnTransformer` from scikit-learn that applies three different transformations to different columns at the same time.

Ordinal encoding is used for columns where categories have a meaningful order. Education for example goes Below College < College < Bachelor < Master < Doctor, which we encode as 1 through 5. The model can then understand that Doctor is higher than Bachelor. Using one-hot encoding here would throw away that order information. This applies to Education, EnvironmentSatisfaction, JobInvolvement, JobLevel, JobSatisfaction, PerformanceRating, RelationshipSatisfaction, StockOptionLevel, and WorkLifeBalance.

One-hot encoding is used for columns where categories have no meaningful order. Department has Sales, R&D, and HR — there is no ranking between them. If we encoded them as 1, 2, 3 the model would think HR is three times Sales, which is wrong. Instead a separate binary column is created for each category. We use `drop='first'` to avoid the dummy variable trap. This applies to BusinessTravel, Department, EducationField, Gender, JobRole, MaritalStatus, and OverTime.

Standard scaling is applied to all numerical columns. MonthlyIncome ranges from 1,000 to 20,000 while Age ranges from 18 to 60. Without scaling, models like Logistic Regression would unfairly weight MonthlyIncome just because the numbers are larger. Standard scaling transforms each column to mean=0 and standard deviation=1 so all features are on the same relative scale. This applies to both original numerical features and the 7 engineered ones.

In [ ]:
# Demonstrate Standard Scaling clearly
from sklearn.preprocessing import StandardScaler

# Example with two columns on very different scales
sample = pd.DataFrame({
    'Age': [25, 35, 45, 55],
    'MonthlyIncome': [2000, 5000, 10000, 20000]
})

scaler = StandardScaler()
scaled = scaler.fit_transform(sample)

print('Before scaling:')
print(sample)
print('\nAfter scaling (both columns are now on the same scale):')
print(pd.DataFrame(scaled, columns=['Age', 'MonthlyIncome']).round(3))

## Step 3 - Applying SMOTE to balance training data

After encoding and scaling, the preprocessor is saved to `artifacts/preprocessor.pkl`. This is important because when making predictions later, we must reuse the exact same preprocessor that was fitted on training data. Fitting a new one on new data would produce different scaling parameters and the predictions would be wrong.

Then SMOTE is applied. SMOTE stands for Synthetic Minority Over-sampling Technique. With about 237 employees who left out of 1176 in training (16%), a model could just always predict "no attrition" and be 84% accurate while being completely useless at identifying who will leave.

SMOTE fixes this by generating synthetic new training examples of the minority class. It takes an existing minority example and creates a new one that is slightly different, based on that example's nearest neighbors in feature space. This is different from simply duplicating rows — the new examples are generated from patterns in the existing data so the model gets a richer view of what attrition looks like.

SMOTE is applied only to training data. The test set stays unchanged so the evaluation reflects the real world.

In [ ]:
# Demonstrate the effect of SMOTE conceptually
print('Before SMOTE (training set):')
print(f'  Total rows: {len(y_train)}')
print(f'  Class 0 (Stayed): {(y_train == 0).sum()}')
print(f'  Class 1 (Left): {(y_train == 1).sum()}')
print(f'  Class 1 percentage: {y_train.mean():.2%}')

print('\nAfter SMOTE (training set - simulated):')
stayed = (y_train == 0).sum()
print(f'  Class 0 (Stayed): {stayed}')
print(f'  Class 1 (Left): {stayed} (matched to majority class)')
print(f'  New total rows: ~{stayed * 2}')
print(f'  Class 1 percentage: ~50%')

print('\nTest set (unchanged by SMOTE):')
print(f'  Class 0: {(y_test == 0).sum()}')
print(f'  Class 1: {(y_test == 1).sum()}')

## Step 4 - Training models with hyperparameter search

Three models are trained, each using a different approach to classification.

Logistic Regression is the simplest classifier and acts as the baseline. It finds a linear decision boundary separating employees who will leave from those who will stay — essentially drawing a straight line through the data. It works well when the relationship between features and the target is approximately linear. The parameter tuned is C, which controls how much the model is penalized for complexity. Lower C means stronger regularization (simpler model) and higher C means it fits the training data more closely. `class_weight='balanced'` is also set to handle remaining imbalance.

Random Forest builds many decision trees, each on a random subset of the data and features, and takes a majority vote across all trees. A single decision tree tends to memorize the training data (overfitting). Averaging many trees that each saw different parts of the data cancels out errors and produces more reliable results. The parameters tuned are the number of trees, how deep each tree can grow, and how much data a node needs before it can split further.

XGBoost is a gradient boosting model that builds trees sequentially. Each new tree focuses on correcting the mistakes of the previous ones. This makes XGBoost very powerful for structured tabular data like this HR dataset and it typically outperforms both Random Forest and Logistic Regression on imbalanced classification. `scale_pos_weight=5.25` is set directly, which is the ratio of stayed-to-left employees. It tells XGBoost to penalize missing an employee who will leave roughly 5 times more heavily than a false alarm.

Every model has hyperparameters that must be set before training. We do not guess these — we search for the best combination. Logistic Regression uses GridSearchCV since the search space is small and every combination can be tried. Random Forest and XGBoost use RandomizedSearchCV, which randomly samples n_iter=5 combinations from a larger parameter space. Both use 3-fold cross-validation, where the training data is split into 3 parts and the model trains on 2 while validating on the 3rd, rotating each time. The average F1-macro across 3 folds is used to rank parameter combinations.

In [ ]:
# Show the hyperparameter configuration from hyperparameters.yaml
import yaml
with open('../config/hyperparameters.yaml', 'r') as f:
    hp = yaml.safe_load(f)

for model_name, config in hp.items():
    print(f'--- {model_name} ---')
    print(f'Search type: {config["search"]}')
    print(f'CV folds: {config["cv"]}')
    print(f'Scoring: {config["scoring"]}')
    if 'param_grid' in config:
        print(f'Param grid: {config["param_grid"]}')
    if 'param_distributions' in config:
        print(f'Param distributions: {config["param_distributions"]}')
        print(f'n_iter: {config["n_iter"]}')
    print()

## Step 5 - Evaluating models

After training, each model is run on the held-out test set and several metrics are calculated. Each one tells us something different.

Accuracy is the percentage of predictions that were correct. But if 84% of employees stayed, a model that always predicts "stay" would score 84% without learning anything useful. This is why accuracy alone is not enough.

F1-macro is the main metric. F1 is the harmonic mean of precision and recall, and the macro version calculates it for each class separately then averages them. This gives equal weight to both the majority class (stayed) and the minority class (left), so a model that ignores attrition cases entirely will get a low score.

Precision measures how many of the employees the model flagged as leaving actually left. Low precision means HR is wasting time interviewing people who were never going to leave.

Recall measures how many of the employees who actually left were correctly identified. Low recall means the model is missing at-risk employees — people who could have been retained.

ROC-AUC measures how well the model separates the two classes across all possible thresholds. A value of 0.5 is random guessing and 1.0 is perfect. This is used as a tiebreaker when two models have the same F1-macro.

FPR (False Positive Rate) is employees who stayed but were predicted to leave — wasted HR interviews. FNR (False Negative Rate) is employees who left but were predicted to stay — missed retention opportunities. Both matter and the summary report shows which model balances them best.

In [ ]:
# Show what the confusion matrix means in this context
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 5))
confusion = np.array([[200, 30], [15, 50]])
im = ax.imshow(confusion, cmap='Blues')

labels = [['True Negative\n(Predicted Stay,\nActually Stayed)',
           'False Positive\n(Predicted Left,\nActually Stayed)\nWasted interview'],
          ['False Negative\n(Predicted Stay,\nActually Left)\nMissed talent',
           'True Positive\n(Predicted Left,\nActually Left)']]

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{confusion[i,j]}\n{labels[i][j]}',
                ha='center', va='center', fontsize=8,
                color='black' if confusion[i,j] < 150 else 'white')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Stay', 'Predicted Left'])
ax.set_yticklabels(['Actually Stayed', 'Actually Left'])
ax.set_title('Confusion Matrix - What each cell means')
plt.tight_layout()
plt.show()

## Step 6 - Saving models, plots, and reports

Each trained model is saved to `artifacts/models/` as a `.pkl` file using `joblib`. This means we do not have to retrain every time we want a prediction — the inference script loads the saved model and runs it directly. The preprocessor is saved to `artifacts/preprocessor.pkl` for the same reason.

Three plots are generated per model: a confusion matrix showing TP, TN, FP, and FN counts, an ROC curve showing how the true positive rate changes against the false positive rate at different thresholds, and a feature importance chart showing which inputs the model relied on most.

After all three models are trained, `save_metrics()` writes a CSV comparing metrics across all models, `plot_model_comparison()` creates a bar chart for easy visual comparison, and `save_summary()` writes a Markdown report to `reports/summary.md`. The best model is selected by highest F1-macro with ROC-AUC as a tiebreaker.

In [ ]:
# Load and display the metrics if they exist
import os

metrics_path = '../reports/metrics/metrics.csv'
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print('Model Comparison Results:')
    print(metrics_df.to_string(index=False))
    
    best = metrics_df.sort_values(['F1-macro', 'ROC-AUC'], ascending=False).iloc[0]
    print(f'\nBest model: {best["Model"]}')
    print(f'F1-macro: {best["F1-macro"]} | ROC-AUC: {best["ROC-AUC"]}')
else:
    print('metrics.csv not found. Run the training pipeline first to generate results.')
    print('Expected at:', metrics_path)

In [ ]:
# Show model comparison chart if metrics exist
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    plot_cols = ['Accuracy', 'F1-macro', 'Precision', 'Recall', 'ROC-AUC']
    
    x = np.arange(len(metrics_df))
    width = 0.15
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, col in enumerate(plot_cols):
        ax.bar(x + i * width, metrics_df[col], width, label=col)
    
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(metrics_df['Model'])
    ax.set_ylim(0, 1.1)
    ax.set_title('Model Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Run the training pipeline to generate metrics and see this chart.')

## The full pipeline in one view

The flow goes from the raw CSV into `load_data()` which drops useless columns, maps Attrition to 0/1, and converts ordinal integers to text labels. Then `validate_data()` checks all required columns exist. `handle_missing_values()` fills numerical gaps with the median and categorical gaps with the mode. `remove_duplicates()` cleans any repeated rows. `handle_outliers()` clips extreme values using IQR on four columns. `engineer_features()` adds the 7 new columns.

After that `train_test_split()` splits into 80/20 with stratification. The ColumnTransformer applies ordinal encoding to ranked categories, one-hot encoding to unranked ones, and standard scaling to all numerical features. SMOTE then balances the training data.

Three models are trained with cross-validated hyperparameter search — Logistic Regression with GridSearchCV, and Random Forest and XGBoost each with RandomizedSearchCV. All three are evaluated on the test set using Accuracy, F1-macro, Precision, Recall, ROC-AUC, FPR, and FNR. Finally the artifacts are saved: models to `artifacts/models/`, the preprocessor to `artifacts/preprocessor.pkl`, plots to `reports/figures/`, metrics to `reports/metrics/`, and a summary to `reports/summary.md`.

## Key design decisions and why

F1-macro is the primary metric because the data is imbalanced and accuracy would be misleading — it gives equal weight to both classes. SMOTE is applied only to training data so synthetic examples never leak into the evaluation, keeping the test set realistic. The preprocessor is saved separately because inference must use the same transformations that were fitted on training data — fitting a new one on new data would give different scaling parameters. Paths and settings are driven by config files so changing a mode or directory only requires editing YAML, not Python code. `lru_cache` on config loaders means the YAML files are read only once per run even if `load_config()` is called many times. `scale_pos_weight` in XGBoost is set to the actual class ratio so it learns to treat false negatives as more costly. 3-fold cross-validation is used instead of 5 because it is faster and the dataset is large enough that 3 folds gives reliable estimates. `stratify=y` in the train-test split ensures both sets have the same class distribution and prevents biased evaluation.